# Step 3.1.2.1: Shape Graphs核心函数定义

## 目标

定义Shape Graphs阵型识别的核心算法函数：
1. 凸包分解函数
2. 垂直层级分配函数
3. 水平位置分配函数
4. 阵型推断函数

## 理论基础

基于Shape Graphs论文（Brandes et al., 2025）的凸包分解方法：
- 使用X坐标（进攻方向）进行前后划分
- 使用Y坐标（边线方向）进行左右划分
- 数据已通过`orient_ball_owning=True`统一为"从左向右进攻"

## 1. 导入库

In [1]:
import numpy as np
from scipy.spatial import ConvexHull

print("✅ 库导入完成")

✅ 库导入完成


## 2. 凸包分解函数

In [2]:
def compute_convex_hull_split(positions):
    """
    计算凸包并找到分割线
    
    参数:
        positions: (N, 2) 球员位置数组
    
    返回:
        split_x: 分割线的X坐标（使用X轴进行前后划分）
    """
    if len(positions) < 3:
        return np.median(positions[:, 0])
    
    try:
        hull = ConvexHull(positions)
        hull_points = positions[hull.vertices]
        
        # 找到凸包的前后边界（X轴方向）
        forward_x = np.max(hull_points[:, 0])  # 前方（进攻方向）
        backward_x = np.min(hull_points[:, 0])  # 后方（防守方向）
        
        # 使用凸包面中心作为分割参考（论文Fig 3的方法）
        # 简化版本：使用X坐标的中位数
        split_x = np.median(hull_points[:, 0])
        
        return split_x
    except:
        # 如果凸包计算失败，使用简单的中位数
        return np.median(positions[:, 0])

print("✅ 凸包分解函数定义完成")

✅ 凸包分解函数定义完成


## 3. 垂直层级分配函数

In [3]:
def assign_vertical_levels(positions, graph):
    """
    将球员分配到垂直层级（F、AM、M、DM、B）
    
    基于论文Fig 3的凸包分解方法
    使用X坐标（进攻方向）进行前后划分
    
    注意：数据在Step 1.1已通过orient_ball_owning=True统一为\"从左向右进攻\"
    因此X坐标大的球员在前方（前锋），X坐标小的球员在后方（后卫）
    
    参数:
        positions: (N, 2) 球员位置数组
        graph: NetworkX图对象
    
    返回:
        levels: 每个球员的垂直层级（0-4，对应B-DM-M-AM-F）
    """
    N = len(positions)
    levels = np.zeros(N, dtype=int)
    
    # 按X坐标排序（从后到前，即从己方球门到对方球门）
    x_coords = positions[:, 0]
    sorted_indices = np.argsort(x_coords)
    
    # 第一次分割：分离前锋和其他球员
    split1 = compute_convex_hull_split(positions)
    forward_mask = x_coords > split1  # X大的是前锋（数据已统一方向）
    backward_mask = ~forward_mask
    
    # 前方部分：前锋（F）和攻击型中场（AM）
    if np.sum(forward_mask) > 0:
        forward_positions = positions[forward_mask]
        forward_indices = np.where(forward_mask)[0]
        
        if len(forward_positions) > 1:
            split_forward = compute_convex_hull_split(forward_positions)
            for i, idx in enumerate(forward_indices):
                if positions[idx, 0] > split_forward:
                    levels[idx] = 4  # F (前锋)
                else:
                    levels[idx] = 3  # AM (攻击型中场)
        else:
            levels[forward_indices] = 4  # F
    
    # 后方部分：中场（M）、防守型中场（DM）、后卫（B）
    if np.sum(backward_mask) > 0:
        backward_positions = positions[backward_mask]
        backward_indices = np.where(backward_mask)[0]
        
        if len(backward_positions) > 2:
            # 第二次分割：分离后卫和中场
            split_backward = compute_convex_hull_split(backward_positions)
            
            for i, idx in enumerate(backward_indices):
                if positions[idx, 0] < split_backward:
                    levels[idx] = 0  # B (后卫)
                else:
                    # 中场区域：再次分割为M和DM
                    mid_mask = (positions[backward_indices, 0] >= split_backward)
                    if np.sum(mid_mask) > 1:
                        mid_positions = positions[backward_indices[mid_mask]]
                        split_mid = np.median(mid_positions[:, 0])
                        if positions[idx, 0] > split_mid:
                            levels[idx] = 2  # M (中场)
                        else:
                            levels[idx] = 1  # DM (防守型中场)
                    else:
                        levels[idx] = 2  # M
        else:
            # 球员太少，简单分配
            for i, idx in enumerate(backward_indices):
                if i < len(backward_indices) // 2:
                    levels[idx] = 0  # B
                else:
                    levels[idx] = 1  # DM
    
    return levels

print("✅ 垂直层级分配函数定义完成")

✅ 垂直层级分配函数定义完成


## 4. 水平位置分配函数

In [4]:
def assign_horizontal_positions(positions, levels):
    """
    在每个垂直层级内分配水平位置（左、中、右）
    
    垂直层级是按X轴划分的，水平位置应该按Y轴划分
    
    参数:
        positions: (N, 2) 球员位置数组
        levels: 垂直层级数组
    
    返回:
        h_positions: 水平位置（0=左, 1=中左, 2=中, 3=中右, 4=右）
    """
    N = len(positions)
    h_positions = np.zeros(N, dtype=int)
    
    # 对每个垂直层级
    for level in range(5):
        level_mask = (levels == level)
        if np.sum(level_mask) == 0:
            continue
        
        level_indices = np.where(level_mask)[0]
        level_y = positions[level_mask, 1]  # 使用Y坐标进行左右划分
        
        # 按Y坐标排序（从左到右）
        sorted_y_indices = np.argsort(level_y)
        n_players = len(sorted_y_indices)
        
        # 分配水平位置
        if n_players == 1:
            h_positions[level_indices[sorted_y_indices[0]]] = 2  # 中
        elif n_players == 2:
            h_positions[level_indices[sorted_y_indices[0]]] = 0  # 左
            h_positions[level_indices[sorted_y_indices[1]]] = 4  # 右
        elif n_players == 3:
            h_positions[level_indices[sorted_y_indices[0]]] = 0  # 左
            h_positions[level_indices[sorted_y_indices[1]]] = 2  # 中
            h_positions[level_indices[sorted_y_indices[2]]] = 4  # 右
        elif n_players == 4:
            h_positions[level_indices[sorted_y_indices[0]]] = 0  # 左
            h_positions[level_indices[sorted_y_indices[1]]] = 1  # 中左
            h_positions[level_indices[sorted_y_indices[2]]] = 3  # 中右
            h_positions[level_indices[sorted_y_indices[3]]] = 4  # 右
        else:  # 5个或更多
            for i, idx in enumerate(sorted_y_indices):
                h_pos = int(i * 4 / (n_players - 1))  # 均匀分布到0-4
                h_positions[level_indices[idx]] = h_pos
    
    return h_positions

print("✅ 水平位置分配函数定义完成")

✅ 水平位置分配函数定义完成


## 5. 阵型推断函数

In [5]:
def identify_shape_graph_edge_players(graph, positions, levels):
    """
    识别Shape Graph边上的球员（用于阵型Code计算）

    根据论文"Codes of shape graphs"部分的定义：
    - Shape graph的边界边（outer edge）上的球员被计入b（后防线）
    - 这些球员可能被垂直层级标记为DM，但在阵型识别中应归入B

    参数:
        graph: NetworkX图对象（Shape Graph）
        positions: (N, 2) 球员位置数组
        levels: 垂直层级数组（0-4对应B-DM-M-AM-F）

    返回:
        on_edge_mask: 布尔数组，标记哪些球员在Shape Graph的边上
    """
    import networkx as nx
    from scipy.spatial import ConvexHull

    N = len(positions)
    on_edge_mask = np.zeros(N, dtype=bool)

    if graph is None or graph.number_of_nodes() == 0:
        return on_edge_mask

    try:
        # 计算凸包找到外边界点
        hull = ConvexHull(positions)
        hull_vertices = set(hull.vertices)

        # 检查Shape Graph的边：如果边的两个端点都在凸包上，则这条边是外边界边
        for edge in graph.edges():
            u, v = edge
            if u in hull_vertices and v in hull_vertices:
                on_edge_mask[u] = True
                on_edge_mask[v] = True

        return on_edge_mask
    except:
        # 凸包计算失败时，使用简化逻辑：X坐标最小的球员视为在边上
        min_x = np.min(positions[:, 0])
        threshold = min_x + 5.0  # 5米阈值
        on_edge_mask = positions[:, 0] <= threshold
        return on_edge_mask


def infer_formation_from_levels(levels, graph=None, positions=None):
    """
    根据垂直层级和Shape Graph边推断阵型

    根据论文"Codes of shape graphs"（Section 5.5）的定义：
    - 阵型Code格式：b(dma)f
      - b: 在shape graph边上的球员（B + 在边上的DM）
      - d: defensive midfielders（不在边上的DM）
      - m: midfielders（M）
      - a: attacking midfielders（AM）
      - f: forwards（F）
    - 最终阵型：将code的字母拼接为数字（如4(003)3 → 433, 4(130)2 → 4132）

    参数:
        levels: 垂直层级数组（0-4对应B-DM-M-AM-F）
        graph: NetworkX图对象（可选，用于识别边上的球员）
        positions: (N, 2) 球员位置数组（可选，用于识别边上的球员）

    返回:
        formation_simple: 简化阵型字符串（如'4-3-3'）
        formation_code: 详细Code（如'4(003)3'）
        formation_detailed: 5位数字格式（如'40033'）
    """
    # 统计每个层级的球员数量
    level_counts = np.bincount(levels, minlength=5)

    # B, DM, M, AM, F
    n_B = level_counts[0]
    n_DM = level_counts[1]
    n_M = level_counts[2]
    n_AM = level_counts[3]
    n_F = level_counts[4]

    # 识别Shape Graph边上的球员
    if graph is not None and positions is not None:
        on_edge_mask = identify_shape_graph_edge_players(graph, positions, levels)

        # 统计在边上的DM数量
        dm_indices = np.where(levels == 1)[0]
        n_DM_on_edge = np.sum(on_edge_mask[dm_indices])
        n_DM_off_edge = n_DM - n_DM_on_edge

        # b = 原始B + 在边上的DM
        b = n_B + n_DM_on_edge
        d = n_DM_off_edge
    else:
        # 如果没有graph信息，使用简化逻辑（向后兼容）
        b = n_B
        d = n_DM

    m = n_M
    a = n_AM
    f = n_F

    # 生成三种格式的阵型
    formation_code = f"{b}({d}{m}{a}){f}"  # 如'4(003)3'
    formation_detailed = f"{b}{d}{m}{a}{f}"  # 如'40033'
    formation_simple = f"{b}-{d+m+a}-{f}"  # 如'4-3-3'

    return formation_simple, formation_code, formation_detailed

print("✅ 阵型推断函数定义完成（新增Shape Graph边识别逻辑）")

✅ 阵型推断函数定义完成（新增Shape Graph边识别逻辑）


## 6. 测试函数

In [6]:
# 测试凸包分解
test_positions = np.array([
    [-40, -10], [-40, 10],  # 后卫
    [-20, -15], [-20, 0], [-20, 15],  # 中场
    [10, -10], [10, 10], [15, 0]  # 前锋
])

split = compute_convex_hull_split(test_positions)
print(f"凸包分割线 X坐标: {split:.2f}")

# 测试垂直层级分配
levels = assign_vertical_levels(test_positions, None)
level_names = ['B', 'DM', 'M', 'AM', 'F']
print(f"\n垂直层级: {[level_names[l] for l in levels]}")

# 测试水平位置分配
h_positions = assign_horizontal_positions(test_positions, levels)
h_pos_names = ['L', 'LC', 'C', 'RC', 'R']
print(f"水平位置: {[h_pos_names[h] for h in h_positions]}")

# 测试新的阵型推断函数（不带graph）
formation_simple, formation_code, formation_detailed = infer_formation_from_levels(levels)
print(f"\n推断阵型（不含Shape Graph边信息）:")
print(f"  简化格式: {formation_simple}")
print(f"  Code格式: {formation_code}")
print(f"  详细格式: {formation_detailed}")

print("\n✅ 所有函数测试通过")

凸包分割线 X坐标: -20.00

垂直层级: ['B', 'B', 'DM', 'DM', 'DM', 'AM', 'AM', 'F']
水平位置: ['L', 'R', 'L', 'C', 'R', 'L', 'R', 'C']

推断阵型（不含Shape Graph边信息）:
  简化格式: 2-5-1
  Code格式: 2(302)1
  详细格式: 23021

✅ 所有函数测试通过


## 总结

本notebook定义了Shape Graphs阵型识别的4个核心函数：

1. ✅ `compute_convex_hull_split()` - 凸包分解
2. ✅ `assign_vertical_levels()` - 垂直层级分配（F/AM/M/DM/B）
3. ✅ `assign_horizontal_positions()` - 水平位置分配（L/LC/C/RC/R）
4. ✅ `infer_formation_from_levels()` - 阵型推断

这些函数将在3.1.2.2中用于批量处理Shape Graphs数据。